[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/laleye/IA-Training-participants/blob/main/module1/session2/notebooks/S2_NB01_signal_spectrogramme.ipynb)

# S2-NB01 — Du signal au spectrogramme

**Module 1 — Session 2 : Traitement du signal audio** · Atelier 2A · 45 min · Ticket `M1-S2-04`

## 🎯 Objectifs
- Inspecter un fichier audio (fréquence d'échantillonnage, canaux, durée, format).
- **Entendre** et **voir** le repliement spectral (*aliasing*) et le bruit de quantification.
- Passer du domaine temporel (forme d'onde) au domaine fréquentiel (spectre, **spectrogramme**).
- Choisir les paramètres d'une STFT et comprendre le compromis temps/fréquence.
- Comprendre ce que perd la **téléphonie** (8 kHz) par rapport à l'ASR (16 kHz).

## 🧭 Comment travailler
| Badge | Pour qui | Attendu |
|---|---|---|
| 🟢 Essentiel | Tout le monde | Exécuter, régler les curseurs, écouter, interpréter (≈ 25 min) |
| 🔵 Intermédiaire | À l'aise en Python | Compléter le code `✍️ À COMPLÉTER` |
| 🔴 Avancé | Experts ML | Défis ouverts |

🎧 **Casque recommandé** : on va beaucoup écouter.

## 0. Préchauffage

In [ ]:
# ⏳ Installation (≈ 1 min sur Colab, instantané ensuite). Ne pas modifier.
import importlib.util, subprocess, sys
if importlib.util.find_spec("m1utils") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "m1utils[all] @ git+https://github.com/laleye/IA-Training-participants.git"], check=True)
import m1utils
INFO = m1utils.setup_notebook()

In [ ]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from m1utils.audio import audio_info, load_audio, naive_downsample, play, quantize, resample, synth_tone
from m1utils.data import data_dir, load_manifest
from m1utils.features import f0_contour
from m1utils.viz import plot_f0, plot_spectrogram, plot_spectrum, plot_waveform

# 📝 PARAMÈTRE : votre dossier TP1 (laisser tel quel pour utiliser les exemples du cours)
MON_DOSSIER_TP1 = "/content/drive/MyDrive/TP1_nom"

demo = load_manifest("demo_s1")
mes_fichiers = sorted((Path(MON_DOSSIER_TP1) / "audio").glob("*.*")) if Path(MON_DOSSIER_TP1, "audio").exists() else []
fichiers = mes_fichiers or [Path(p) for p in demo.abspath] + sorted((data_dir() / "demo_s1" / "audio").glob("*_8k.wav"))
print(f"{len(fichiers)} fichiers ({'vos enregistrements TP1' if mes_fichiers else 'exemples du cours'})")

## 1. Inspecter un fichier audio 🟢
Avant de traiter un fichier, on regarde sa **carte d'identité** : fréquence d'échantillonnage (*sampling rate*, Hz),
nombre de canaux, durée, format et profondeur de quantification (bits).

In [ ]:
inventaire = pd.DataFrame([audio_info(f) for f in fichiers])
inventaire["fichier"] = inventaire["path"].map(lambda p: Path(p).name)
display(inventaire[["fichier", "sr", "channels", "duration_s", "format", "subtype", "bit_depth"]].round(2))

**❓ Question :** repérez les fichiers `*_8k.wav`. D'où viennent-ils et quelle est leur fréquence d'échantillonnage ?
Pourquoi les modèles du cours attendent-ils 16 kHz ?

✍️ *Votre réponse :* …

## 2. Échantillonnage et repliement 🟢
Numériser un son, c'est le **mesurer** `fs` fois par seconde. Le **théorème de Nyquist-Shannon** dit qu'on ne peut
représenter que les fréquences **inférieures à fs/2** (la fréquence de Nyquist). Au-delà, la fréquence se **replie** :
elle réapparaît, fausse, plus bas.

Écoutez une sinusoïde de 7 000 Hz échantillonnée à 16 kHz, puis réduite à 8 kHz de deux façons :
**naïvement** (on garde un échantillon sur deux) ou **proprement** (filtre anti-repliement, puis réduction).

In [ ]:
original = synth_tone(7_000, duration_s=1.5, sr=16_000)
naif = naive_downsample(original, 2)       # 8 kHz sans filtre
propre = resample(original, 16_000, 8_000)  # 8 kHz avec filtre anti-repliement

play(original, 16_000, label="Original : 7 000 Hz (fs = 16 kHz)")
play(naif, 8_000, label="Réduction NAÏVE à 8 kHz", normalize=False)
play(propre, 8_000, label="Réduction PROPRE à 8 kHz (filtre anti-repliement)", normalize=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.2), sharey=True)
for ax, (signal, fs, titre) in zip(axes, [(original, 16_000, "original 16 kHz"), (naif, 8_000, "naïf 8 kHz"),
                                          (propre, 8_000, "propre 8 kHz")]):
    plot_spectrum(signal[:4000], fs, ax=ax, max_freq=8_000, title=titre)
plt.tight_layout()
plt.show()

**❓ Questions :** quelle fréquence entendez-vous dans la version naïve ? Que devient le son dans la version propre ?

✍️ *Votre réponse :* …

### Exercice 2.1 🔵 — Prédire la fréquence repliée
Pour une fréquence `f` échantillonnée à `fs`, la fréquence apparente est la distance de `f` au multiple de `fs` le plus
proche : `|f - fs * round(f / fs)|`. Complétez la fonction, puis vérifiez avec le curseur ci-dessous.

In [ ]:
def frequence_apparente(f, fs):
    return ...  # ✍️ À COMPLÉTER


assert frequence_apparente(7_000, 8_000) == 1_000, "7 kHz à fs = 8 kHz doit apparaître à 1 kHz"
assert frequence_apparente(3_000, 8_000) == 3_000, "3 kHz < Nyquist : inchangé"
print("✅ Fonction correcte")


def explorer_repliement(f=7000, fs=8000):
    signal = naive_downsample(synth_tone(f, 1.0, sr=48_000), 48_000 // fs)
    fig, ax = plt.subplots(figsize=(9, 2.8))
    plot_spectrum(signal[:4000], fs, ax=ax, max_freq=fs / 2,
                  title=f"f = {f} Hz, fs = {fs} Hz → apparente {frequence_apparente(f, fs):.0f} Hz")
    plt.show()


widgets.interact(explorer_repliement, f=widgets.IntSlider(7000, 100, 15000, 100),
                 fs=widgets.Dropdown(options=[8000, 16000, 24000], value=8000));

## 3. Quantification 🔵
Chaque mesure est arrondie à l'un des 2ⁿ niveaux possibles (n = **bits**). Moins de bits = arrondi plus grossier =
**bruit de quantification**. Écoutez une phrase en 4, 8 et 16 bits.

In [ ]:
y, sr = load_audio(demo[demo.lang == "wol"].abspath.iloc[0])
for bits in (4, 8, 16):
    yq = quantize(y, bits)
    snr = 10 * np.log10(np.mean(y**2) / np.mean((y - yq) ** 2))
    play(yq, sr, label=f"{bits} bits — SNR de quantification ≈ {snr:.0f} dB", normalize=False)

🔴 **Défi** : tracez le SNR de quantification en fonction du nombre de bits (2 à 16) et vérifiez la règle des
« ≈ 6 dB par bit ». Pourquoi le 16 bits (≈ 96 dB de dynamique théorique) suffit-il largement pour la parole ?

In [ ]:
...  # ✍️ À COMPLÉTER

## 4. La forme d'onde 🟢
Le signal brut : l'amplitude (pression de l'air) au cours du temps. On y voit les **silences** et les **syllabes**,
mais presque rien sur les **sons** eux-mêmes.

In [ ]:
choix = fichiers[0]
y, sr = load_audio(choix)
play(y, sr, label=choix.name)
plot_waveform(y, sr, title=f"Forme d'onde — {choix.name}")
plt.show()

## 5. Le spectre d'une voyelle 🔵
La **transformée de Fourier** (FFT) décompose un son en **fréquences**, comme un prisme décompose la lumière.
Sur 40 ms au cœur d'une voyelle, on voit des **harmoniques** régulièrement espacées (multiples de la F0) et une
enveloppe avec des bosses : les **formants**.

Complétez pour extraire 40 ms autour de l'instant où l'énergie est maximale.

In [ ]:
trame = int(0.04 * sr)
energie = np.convolve(y**2, np.ones(trame), mode="same")
centre = ...                # ✍️ À COMPLÉTER : indice de l'énergie maximale
segment = ...               # ✍️ À COMPLÉTER : 40 ms centrées sur `centre`
assert len(segment) == trame, "Le segment doit durer 40 ms"
plot_spectrum(segment, sr, max_freq=4000, title="Spectre d'un segment voisé de 40 ms")
_, f0 = f0_contour(y, sr)
plt.axvline(np.nanmedian(f0), color="#e07a1f", ls="--", label=f"F0 médiane ≈ {np.nanmedian(f0):.0f} Hz")
plt.legend()
plt.show()

**❓ Question :** quel est l'écart entre deux pics successifs ? Correspond-il à la F0 ? Où sont les formants ?

✍️ *Votre réponse :* …

## 6. Le spectrogramme : le temps ET la fréquence 🟢
La parole change sans cesse : on calcule donc un spectre sur de **courtes fenêtres** (≈ 25 ms) qui glissent
(pas ≈ 10 ms) : c'est la **transformée de Fourier à court terme** (STFT). Empilées, elles forment le **spectrogramme** :
temps en abscisse, fréquence en ordonnée, énergie en couleur.

Jouez avec la taille de fenêtre `n_fft` : **petite** = bonne précision temporelle (bande large) ;
**grande** = bonne précision fréquentielle (bande étroite, on voit les harmoniques).

In [ ]:
def explorer_stft(n_fft=512, hop=160):
    fig, ax = plt.subplots(figsize=(11, 3.5))
    plot_spectrogram(y, sr, n_fft=n_fft, hop_length=hop, ax=ax, max_freq=5000,
                     title=f"n_fft = {n_fft} ({1000 * n_fft / sr:.0f} ms) · hop = {hop} ({1000 * hop / sr:.0f} ms)")
    plt.show()


widgets.interact(explorer_stft, n_fft=widgets.Dropdown(options=[128, 256, 400, 512, 1024, 2048], value=512),
                 hop=widgets.Dropdown(options=[80, 160, 320], value=160));

### Exercice 6.1 🔵 — Le compromis temps / fréquence
Remplissez le tableau : pour chaque `n_fft` (à fs = 16 kHz), la **durée de la fenêtre** (ms) et la **résolution
fréquentielle** Δf = fs / n_fft (Hz). Lequel permet de distinguer des harmoniques espacées de 120 Hz ?

In [ ]:
tailles = [128, 400, 1024, 2048]
tableau = pd.DataFrame({"n_fft": tailles, "fenêtre_ms": ..., "delta_f_Hz": ...})  # ✍️ À COMPLÉTER
display(tableau)

✍️ *Votre réponse :* …

## 7. Comparer les deux mots d'une paire minimale 🟢
Spectrogramme + F0 côte à côte : ce qui change (la F0 pour un ton, la durée pour une voyelle longue) et ce qui ne
change pas (les formants).

In [ ]:
paires = load_manifest("minimal_pairs")
paire = paires[paires["pair"] == "P4"].groupby("lang").head(2)
fig, axes = plt.subplots(1, len(paire), figsize=(6 * len(paire), 3.4))
for ax, (_, ligne) in zip(np.atleast_1d(axes), paire.iterrows()):
    signal, fs = load_audio(ligne.abspath)
    plot_f0(signal, fs, ax=ax, title=f"[{ligne.lang}] {ligne.text} — {ligne.contrast}")
plt.tight_layout()
plt.show()

## 8. La téléphonie : 8 kHz 🟢
Comparez un énoncé téléphonique d'origine (8 kHz) et sa version rééchantillonnée à 16 kHz.

In [ ]:
tel = sorted((data_dir() / "demo_s1" / "audio").glob("*_8k.wav"))[0]
y8, sr8 = load_audio(tel, sr=None)
y16 = resample(y8, sr8, 16_000)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.4))
plot_spectrogram(y8, sr8, ax=axes[0], max_freq=8000, title="Original téléphone (8 kHz)")
plot_spectrogram(y16, 16_000, ax=axes[1], max_freq=8000, title="Rééchantillonné à 16 kHz")
plt.tight_layout()
plt.show()

**❓ Question :** que voit-on au-dessus de 4 kHz après rééchantillonnage ? Quels sons du fongbe ou du wolof risquent
d'être moins bien reconnus au téléphone ?

✍️ *Votre réponse :* …

### Défi 8.1 🔴 — L'effet d'un codec avec perte
Si `ffmpeg` est disponible, encodez un énoncé en MP3 à 16 kbit/s, décodez-le et comparez les spectrogrammes.
Qu'ont supprimé l'encodeur et son **modèle psychoacoustique** ?

In [ ]:
...  # ✍️ À COMPLÉTER

## ✅ Bilan
- Un fichier audio = des **mesures** à `fs` Hz, sur n **bits** ; on représente au plus fs/2 (**Nyquist**).
- Réduire `fs` sans filtre **invente** des fréquences (repliement) : toujours rééchantillonner proprement.
- Le **spectre** montre la F0 (harmoniques) et les formants ; le **spectrogramme** les montre au cours du temps.
- Le réglage **25 ms / 10 ms** est le compromis standard pour la parole.
- La **téléphonie** (8 kHz) perd tout au-dessus de 4 kHz : une raison majeure de dégradation de l'ASR sur le terrain.

**Prochaine étape :** cours 2B — échelle Mel, MFCC et prétraitement ; puis TD2 (lecture de spectrogrammes).